# Data Leakage in Machine Learning

Data leakage is one of the most dangerous and misunderstood problems in machine learning. It usually does not produce errors or warnings. Instead, it quietly creates excellent validation scores and impressive cross-validation results that collapse in the real world.

## What Is Data Leakage?

Data leakage occurs when a model uses information during training that would not be available at prediction time in the real world. In other words, the model learns from future data, target-derived signals, or improperly shared information between training and test sets.

The core rule is simple: a model must only learn from information that would exist at the moment a real-world prediction is made.

## Why Leakage Is Dangerous

Leakage is dangerous because it produces artificially high performance, hides generalization failure, misleads stakeholders, wastes engineering effort, and damages trust when deployed systems fail. A model achieving 98% accuracy may not be brilliant, it may simply be cheating.

Leakage does not make your training result better in a meaningful way. It makes your evaluation dishonest.

## Example 1: Target Leakage

Imagine a churn prediction problem with features like Tenure, MonthlyCharges, ContractType, SupportCallsLast90Days, Churn, and CancellationReason. If you include CancellationReason as a feature, that is leakage because it only exists after a customer has already churned.

Rule: if a feature would not exist before the outcome happens, it is leakage.

## Example 2: Temporal Leakage

Suppose you predict whether a patient will be readmitted within 30 days of discharge. A feature such as FollowUpAppointmentBooked may be leakage if it is created after the prediction moment or if it reflects downstream decisions influenced by the outcome process.

Rule: if a feature reflects something that happens after the prediction decision point, it is leakage.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
df = pd.DataFrame({
    'tenure': np.random.exponential(scale=20, size=1000),
    'monthly_charges': np.random.gamma(shape=2, scale=30, size=1000),
    'cancellation_reason': np.random.choice([0, 1], size=1000, p=[0.8, 0.2]),
    'churn': np.random.choice([0, 1], size=1000, p=[0.73, 0.27])
})

X = df[['tenure', 'monthly_charges', 'cancellation_reason']]
y = df['churn']

print('Example dataset shape:', df.shape)
print('Potential leakage feature included:', 'cancellation_reason' in X.columns)

## Example 3: Train-Test Contamination

A common mistake is to fit preprocessing on the full dataset before splitting. For example, scaling all numerical columns before the train-test split allows the scaler to use information from the test set. That leaks test distribution statistics into training.

Correct workflow: split first, fit on training data only, then transform the test set using the fitted transformer.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X[['tenure', 'monthly_charges']],
    y,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('Scaler fitted only on training data:', True)

## How to Detect Leakage

Watch for suspiciously high accuracy, near-perfect correlation between a feature and the target, model performance that collapses in production, or features that seem too good to be true. Leakage is usually a conceptual boundary violation, not a syntax error.

The best mental test is simple: at the exact moment a real prediction is made, would you actually know this feature?

## Types of Leakage

There are three main categories:

- Target leakage: features derived from or dependent on the target
- Temporal leakage: future information used before it would exist
- Train-test contamination: test information influences training through preprocessing or aggregation

Each one violates the same rule: the model must only learn from information available at prediction time.

## Prevention Checklist

- Separate X and y early
- Split into train and test before fitting anything
- Fit preprocessing only on training data
- Avoid features derived from the target
- Avoid future information
- Review every feature with the prediction moment test
- Document feature availability assumptions
- Perform sanity checks on correlations

Leakage prevention is discipline, not luck.

## Closing Reflection

Data leakage is not a coding error. It is a thinking error. Every feature must pass the prediction moment test, every preprocessing step must respect the train-test boundary, and every evaluation must be honest.

If the model sees information it would not have in real life, your evaluation is invalid. Guard the boundary. Protect the test set. Build honest models.

## Bonus Content 🎁

This section is optional, and learners who want to explore the topics covered so far can utilize the materials provided below.

- Kaggle Tutorial on Data Leakage
- Google Machine Learning Crash Course – Data Preparation